# Use Case 4: Multi-Class Real-Image Classification using CNN

This notebook classifies astronaut, cat, coffee and rocket images.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

CNN and visualization libraries are imported.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)


## Step 2: Configure dataset

Four class folders are included.

In [ ]:
DATASET_PATH = "../datasets/02_multiclass_objects"
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 16


## Step 3: Load and label images

Class labels are inferred from folder names.

In [ ]:
DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 4: Display samples

Sample images from the training data are shown.

In [ ]:
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for index in range(min(9, len(images))):
    plt.subplot(3, 3, index + 1)
    plt.imshow(images[index].numpy().astype("uint8"))
    plt.title(class_names[int(labels[index])])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 5: Build the multi-class CNN

The softmax output produces one probability for every class.

In [ ]:
model = Sequential([
    Input(shape=(128, 128, 3)),
    Rescaling(1.0 / 255),

    Conv2D(32, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(64, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(128, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(len(class_names), activation="softmax"),
])


## Step 6: Compile and train

Sparse categorical cross-entropy supports multiple integer-labelled classes.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)


## Step 7: Evaluate

Training and validation curves are compared.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()
